# CardioScope-XAI — XGBoost Classifier
Trains the tabular risk classifier on UCI Cleveland features, evaluates performance, and logs everything to MLflow.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data_loader import load_and_clean
from src.feature_engineering import build_feature_matrix
from src.xgboost_model import train, evaluate, cross_validate_model, DEFAULT_PARAMS

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

## 1. Load & Prepare Data

In [ ]:
df = load_and_clean('../data/raw/heart_disease_uci.csv')
X, scaler = build_feature_matrix(df, fit_scaler=True)
y = df['target'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Features: {X.shape[1]}')

## 2. Train XGBoost + Log to MLflow

In [ ]:
model, metrics = train(X_train, y_train, X_test, y_test, log_mlflow=True)

print('\n── Test Set Metrics ──')
for k, v in metrics.items():
    print(f'  {k:12s}: {v}')

## 3. Full Evaluation Report

In [ ]:
full_metrics = evaluate(model, X_test, y_test)

## 4. Confusion Matrix

In [ ]:
cm = np.array(full_metrics['confusion_matrix'])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['No Disease', 'Disease'],
    yticklabels=['No Disease', 'Disease'],
    linewidths=0.5
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — XGBoost', fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives : {tn}  |  False Positives: {fp}')
print(f'False Negatives: {fn}  |  True Positives : {tp}')
print(f'\nFalse Negative Rate (missed disease): {fn/(fn+tp):.2%}')

## 5. Cross-Validation (5-Fold Stratified)

In [ ]:
cv_results = cross_validate_model(X, y, n_splits=5)

print('── 5-Fold Cross-Validation ──')
for metric, vals in cv_results.items():
    print(f'  {metric:12s}: {vals["mean"]:.4f} ± {vals["std"]:.4f}')

## 6. Feature Importance

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(importance)))
importance.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('XGBoost Feature Importance (gain)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\nTop 10 most important features:')
print(importance.tail(10).sort_values(ascending=False).to_string())

## 7. ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

y_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, color='#1565C0', lw=2, label=f'XGBoost (AUC = {full_metrics["auc_roc"]:.4f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
ax.fill_between(fpr, tpr, alpha=0.1, color='#1565C0')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve — XGBoost', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Model Summary

| Metric | Value |
|---|---|
| Accuracy | See output above |
| Recall (sensitivity) | Priority metric — minimise false negatives |
| AUC-ROC | Threshold-independent performance |
| Model saved | `models/xgboost_model.pkl` |
| MLflow logged | `mlflow/` directory |